<a href="https://colab.research.google.com/github/Sevin-Tayfehaliverdi/bioinformatics-lab/blob/main/Bio6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re
from Bio import Entrez, SeqIO

Entrez.email = "your.email@example.com"
ACCESSION = "L09137"

handle = Entrez.efetch(db="nucleotide", id=ACCESSION, rettype="fasta", retmode="text")
record = SeqIO.read(handle, "fasta")
handle.close()

seq = str(record.seq).upper()
print(f"Accession : {record.id}")
print(f"Length    : {len(seq)} nt")
assert 1000 <= len(seq) <= 3000, "Sequence must be between 1000 and 3000 nucleotides!"

enzymes = {
    "EcoRI":   {"site": "GAATTC", "cut_pos": 1, "overhang": "5' AATT"},
    "BamHI":   {"site": "GGATCC", "cut_pos": 1, "overhang": "5' GATC"},
    "HindIII": {"site": "AAGCTT", "cut_pos": 1, "overhang": "5' AGCT"},
}

def cleavage_notation(site, cut_pos):
    return f"5'-{site[:cut_pos]}^{site[cut_pos:]}-3'"

results = {}
for name, info in enzymes.items():
    site = info["site"]

    starts = [m.start() for m in re.finditer(f"(?={site})", seq)]
    cuts = [s + info["cut_pos"] for s in starts]
    results[name] = cuts

    print("\n" + "=" * 60)
    print(f"Enzyme            : {name}")
    print(f"Recognition site  : 5'-{site}-3'")
    print(f"Cleavage site     : {cleavage_notation(site, info['cut_pos'])}")
    print(f"Sticky end        : {info['overhang']} overhang")
    print(f"Number of sites   : {len(starts)}")
    if starts:
        print(f"Site start (1-based): {[s + 1 for s in starts]}")
        print(f"Cut after position  : {cuts}")

        points = [0] + sorted(cuts) + [len(seq)]
        frags = [points[i + 1] - points[i] for i in range(len(points) - 1)]
        print(f"Fragments (linear)  : {frags}")
    else:
        print("No cleavage sites found.")

all_cuts = sorted(c for cuts in results.values() for c in cuts)
points = [0] + all_cuts + [len(seq)]
frags = [points[i + 1] - points[i] for i in range(len(points) - 1)]
print("\n" + "=" * 60)
print("Triple digest (EcoRI + BamHI + HindIII), linear fragments:", frags)

def show_context(name, flank=10):
    site = enzymes[name]["site"]
    for m in re.finditer(f"(?={site})", seq):
        s = m.start()
        c = s + enzymes[name]["cut_pos"]
        left = seq[max(0, s - flank):c]
        right = seq[c:s + len(site) + flank]
        print(f"{name}: ...{left} ✂ {right}...")

print("\nCut sites in context (✂ = cut on top strand):")
for n in enzymes:
    show_context(n)

Accession : L09137.2
Length    : 2686 nt

Enzyme            : EcoRI
Recognition site  : 5'-GAATTC-3'
Cleavage site     : 5'-G^AATTC-3'
Sticky end        : 5' AATT overhang
Number of sites   : 1
Site start (1-based): [396]
Cut after position  : [396]
Fragments (linear)  : [396, 2290]

Enzyme            : BamHI
Recognition site  : 5'-GGATCC-3'
Cleavage site     : 5'-G^GATCC-3'
Sticky end        : 5' GATC overhang
Number of sites   : 1
Site start (1-based): [417]
Cut after position  : [417]
Fragments (linear)  : [417, 2269]

Enzyme            : HindIII
Recognition site  : 5'-AAGCTT-3'
Cleavage site     : 5'-A^AGCTT-3'
Sticky end        : 5' AGCT overhang
Number of sites   : 1
Site start (1-based): [447]
Cut after position  : [447]
Fragments (linear)  : [447, 2239]

Triple digest (EcoRI + BamHI + HindIII), linear fragments: [396, 21, 30, 2239]

Cut sites in context (✂ = cut on top strand):
EcoRI: ...GACGGCCAGTG ✂ AATTCGAGCTCGGTA...
BamHI: ...CGGTACCCGGG ✂ GATCCTCTAGAGTCG...
HindIII: ...GCA